# Formation Energy per Atom Prediction for Crystal Materials: End-to-End Inference Example Using Matformer

## Model Overview: What is Matformer?

**Matformer** is a Graph Neural Network (GNN) specifically designed for **crystal materials**. Its core idea is to model crystal structures as an **atom-bond graph**, learning interactions between atoms to predict physicochemical properties of materials.

### Key Features:

- **Graph-based Modeling**:  
  Each atom is treated as a node, chemical bonds between atoms as edges, and lattice vectors as global features, forming a **periodic graph network**.

- **Multi-scale Attention Mechanism**:  
  Attention weights are used to automatically learn which atoms or bonds have greater influence on the target property (e.g., formation energy).

- **High Scalability**:  
  Supports multiple materials databases (e.g., Materials Project, OQMD) and can predict various properties such as formation energy, band gap, and elastic modulus.

> Reference Paper:  
> "Periodic Graph Transformers for Crystal Material Property Prediction"

This notebook demonstrates how to use the Matformer model to perform inference on **formation energy per atom** of crystal materials and visualize the predicted results along with crystal structures.

In [ ]:
!pip install pymatgen matplotlib ase numpy pydantic==1.10.9 jarvis-tools==2022.9.16 pathos

In [ ]:
import os
import pickle
import yaml
import numpy as np
import mindspore as ms
from mindspore import set_seed
from mindspore import nn
from mindspore.amp import all_finite
from mindchemistry.cell.matformer.matformer import Matformer
from mindchemistry.cell.matformer.utils import LossRecord, OneCycleLr
from mindchemistry.graph.loss import L1LossMask, L2LossMask
from mindchemistry.graph.dataloader import DataLoaderBase as DataLoader

config_path = "config.yaml"
with open(config_path, 'r', encoding='utf-8') as stream:
    config = yaml.safe_load(stream)

device_target = config['train']["device"]  # "Ascend"
device_id = config['train']["device_id"]
ms.set_context(device_target=device_target, device_id=device_id)
set_seed(config['train']["seed"])

## Dataset Description: `jdft_3d-12-12-2022.json`

### Basic Information

- **Dataset Name**: `jdft_3d-12-12-2022.json`
- **Source**: [JARVIS-DFT](https://jarvis.nist.gov/) (Joint Automated Repository for Various Integrated Simulations - Density Functional Theory)
- **Size**: 75,993 3D crystal structures
- **Format**: JSON
- **Material Identifier**: Unique ID using `jid` (e.g., `JVASP-90856`)

---

### Data Overview

This dataset contains **3D bulk crystal materials** structures and properties computed using Density Functional Theory (DFT). It is suitable for materials discovery, property prediction, and machine learning modeling tasks.

---

### Key Fields

| Field | Type | Description |
|-------|------|-------------|
| `jid` | str | Unique JARVIS material ID (e.g., `JVASP-90856`) |
| `formula` | str | Chemical formula (e.g., `TiCuSiAs`) |
| `spg_number` / `spg_symbol` | int / str | Space group number and symbol (e.g., 129, `P4/nmm`) |
| `formation_energy_peratom` | float | Formation energy per atom (eV/atom), more negative indicates higher stability |
| `optb88vdw_bandgap` | float | Band gap calculated with OptB88vdW functional (eV) |
| `mbj_bandgap`, `hse_gap` | float | Band gap from mBJ or HSE06 functional (available for some materials) |
| `atoms` | dict | **Core structure field**, including:<br>• `lattice_mat`: lattice matrix (3×3)<br>• `coords`: atomic coordinates<br>• `elements`: list of elements<br>• `cartesian`: whether coordinates are Cartesian (bool) |
| `density` | float | Material density (g/cm³) |
| `ehull` | float | Energy above convex hull (eV/atom), <0.1 eV/atom considered stable |
| `func` | str | DFT functional used (e.g., `OptB88vdW`) |
| `dimensionality` | str | Material dimensionality (all 3D-bulk in this dataset) |
| `crys` | str | Crystal system (e.g., `tetragonal`, `cubic`) |
| `nat` | int | Total number of atoms |
| `reference` | str | Corresponding Materials Project ID (e.g., `mp-1080455`) |

---

### Dataset Construction

The main target of this project is **`formation_energy_peratom`**.  

- **Input**: Atoms + lattice information → graph structure  
- **Output**: Target property (`formation_energy_peratom`)  

The model does **not** directly use the properties as input. Instead, the crystal graph is constructed from the `atoms` field. If you want to use a custom dataset, the `atoms` field is mandatory.

Minimal required fields for custom datasets:

```json
[
  {
    "jid": "EXAMPLE-0001",
    "nat": 4,
    "atoms": {
      "abc": [3.56693, 3.56693, 9.39708],
      "angles": [90.0, 90.0, 90.0],
      "cartesian": true,
      "coords": [
        [2.6751975, 2.6751975, 7.37610175],
        [0.8917325, 0.8917325, 2.02097824],
        [0.8917325, 2.6751975, 4.69854],
        [2.6751975, 0.8917325, 4.69854],
        [0.8917325, 2.6751975, 0.0],
        [2.6751975, 0.8917325, 0.0],
        [2.6751975, 2.6751975, 2.88947956],
        [0.8917325, 0.8917325, 6.50760044]
      ],
      "elements": ["Ti","Ti","Cu","Cu","Si","Si","As","As"],
      "lattice_mat": [
        [3.566933224304235, 0.0, -0.0],
        [0.0, 3.566933224304235, -0.0],
        [-0.0, -0.0, 9.397075454186664]
      ],
      "props": ["","","","","","","",""]
    },
    "formation_energy_peratom": -1.23
  }
]
```

For other prediction tasks, simply replace `formation_energy_peratom` with another target property, e.g., `optb88vdw_bandgap`:

```json
{
  "jid": "CUSTOM-00002",
  "nat": 4,
  "atoms": {...},
  "optb88vdw_bandgap": 1.73
}
```

In [ ]:
from data.generate import get_prop_model
# mkdir
dataset_dir = config['train']["dataset_dir"]
ckpt_dir = config['train']["ckpt_dir"]
os.makedirs(dataset_dir, exist_ok=True)
os.makedirs(ckpt_dir, exist_ok=True)

get_prop_model(prop=config['train']["props"], use_lattice=True, dataset_path=config["dataset"])

In [ ]:
# Load training data
with open(config['dataset']['x_train_path'], 'rb') as f:
    x = pickle.load(f)
with open(config['dataset']['edge_index_train_path'], 'rb') as f:
    edge_index = pickle.load(f)
with open(config['dataset']['edge_attr_train_path'], 'rb') as f:
    edge_attr = pickle.load(f)
with open(config['dataset']['label_train_path'], 'rb') as f:
    label = pickle.load(f)

# Load validation data
with open(config['dataset']['x_val_path'], 'rb') as f:
    x_val = pickle.load(f)
with open(config['dataset']['edge_index_val_path'], 'rb') as f:
    edge_index_val = pickle.load(f)
with open(config['dataset']['edge_attr_val_path'], 'rb') as f:
    edge_attr_val = pickle.load(f)
with open(config['dataset']['label_val_path'], 'rb') as f:
    label_val = pickle.load(f)

# Initialize model
matformer = Matformer(config['model'])
model_parameters = filter(lambda p: p.requires_grad, matformer.get_parameters())
params = sum(np.prod(p.shape) for p in model_parameters)
print("Model trainable parameters: %s", params)

# Optimizer configuration (weight decay grouping)
no_decay_params = list(filter(lambda x: 'bias' in x.name or 'norm' in x.name or 'bn' in x.name, matformer.trainable_params()))
decay_params = list(filter(lambda x: 'bias' not in x.name and 'norm' not in x.name and 'bn' not in x.name, matformer.trainable_params()))
group_params = [{'params': decay_params, 'weight_decay': 0.00001},
                {'params': no_decay_params, 'weight_decay': 0},
                {'order_params': matformer.trainable_params()}]
optimizer = nn.AdamWeightDecay(params=group_params, eps=1e-8)
optimizer.beta1 = ms.Tensor([0.95], ms.float32)

# Loss functions
loss_func_mse = L2LossMask(reduction='mean')
loss_func_mae = L1LossMask(reduction='mean')

# Define forward pass
def forward(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size):
    pred = matformer(data_x, data_edge_attr, data_edge_index, data_batch, node_mask, edge_mask, node_num)
    mseloss = loss_func_mse(pred, data_label, num=batch_size)
    return mseloss, pred

# Backward pass
backward = ms.value_and_grad(forward, None, weights=matformer.trainable_params(), has_aux=True)

# Training step
def train_step(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size):
    (mseloss, pred), grads = backward(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size)
    is_finite = all_finite(grads)
    if is_finite:
        optimizer(grads)
    return mseloss, is_finite, pred

# Validation forward pass
def forward_eval(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size):
    pred = matformer(data_x, data_edge_attr, data_edge_index, data_batch, node_mask, edge_mask, node_num)
    mseloss = loss_func_mse(pred, data_label, num=batch_size)
    maeloss = loss_func_mae(pred, data_label, num=batch_size)
    return mseloss, maeloss, pred

def eval_step(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size):
    return forward_eval(data_x, data_edge_attr, data_edge_index, data_batch, data_label, node_mask, edge_mask, node_num, batch_size)

# Create data loaders
BATCH_SIZE = config['train']['batch_size']
train_loader = DataLoader(BATCH_SIZE, edge_index, node_attr=x, edge_attr=edge_attr, label=label, dynamic_batch_size=True)
eval_loader = DataLoader(BATCH_SIZE, edge_index_val, node_attr=x_val, edge_attr=edge_attr_val, label=label_val, dynamic_batch_size=False)

# Learning rate scheduler
one_cycle_lr_cls = OneCycleLr(steps_per_epoch=int(len(label) / BATCH_SIZE), epochs=config['train']["epoch_size"], max_lr=0.001, optimizer=optimizer)

# Check for existing checkpoint
if os.path.exists(config['checkpoint']['best_loss_path']):
    print("Loading existing checkpoint from: %s", config['checkpoint']['best_loss_path'])
    param_dict = ms.load_checkpoint(config['checkpoint']['best_loss_path'])
    epoch = int(param_dict["epoch"]) + 1
    param_not_load, _ = ms.load_param_into_net(matformer, param_dict)
    one_cycle_lr_cls.current_step = int(param_dict["current_step"])
    one_cycle_lr_cls.step()
    print("Resuming training from epoch: %d", epoch)
else:
    print("Starting new training process")
    epoch = 0

# Main training loop
BEST_EPOCH_EVAL_MSE_LOSS = float('inf')
for epoch in range(epoch, config['train']["epoch_size"]):
    # Training
    matformer.set_train(True)
    train_mseloss_record = LossRecord()
    for batch in train_loader:
        (x, edge_attr, y, edge_idx, batch_idx, node_mask, edge_mask, node_num, bs) = batch
        mseloss, is_finite, _ = train_step(x, edge_attr, edge_idx, batch_idx, y, node_mask, edge_mask, node_num, bs)
        train_mseloss_record.update(mseloss)
    # Validation
    matformer.set_train(False)
    eval_mseloss_record = LossRecord()
    eval_maeloss_record = LossRecord()
    for batch in eval_loader:
        (x, edge_attr, y, edge_idx, batch_idx, node_mask, edge_mask, node_num, bs) = batch
        mseloss, maeloss, _ = eval_step(x, edge_attr, edge_idx, batch_idx, y, node_mask, edge_mask, node_num, bs)
        eval_mseloss_record.update(mseloss)
        eval_maeloss_record.update(maeloss)
    # Save best model
    if eval_mseloss_record.avg < BEST_EPOCH_EVAL_MSE_LOSS:
        BEST_EPOCH_EVAL_MSE_LOSS = eval_mseloss_record.avg
        state = {"epoch": str(epoch), "current_step": str(one_cycle_lr_cls.current_step)}
        ms.save_checkpoint(matformer, config['checkpoint']['best_loss_path'], append_dict=state)
        print("Saved best model at epoch %d, MSE: %.6f", epoch, eval_mseloss_record.avg)

    print(f"Epoch {epoch} | Train MSE: {train_mseloss_record.avg.item():.6f} | Val MSE: {eval_mseloss_record.avg.item():.6f} | Val MAE: {eval_maeloss_record.avg.item():.6f}")
print("Training completed. Best model saved to: %s", config['checkpoint']['best_loss_path'])

In [ ]:
# Print model parameters
model_parameters = filter(lambda p: p.requires_grad, matformer.get_parameters())
params = sum(np.prod(p.shape) for p in model_parameters)
print("The model has %s parameters." % params)

# Define loss functions
loss_func_mse = L2LossMask(reduction='mean')
loss_func_mae = L1LossMask(reduction='mean')

# Load checkpoint
checkpoint_dir = config['predictor']['checkpoint_path']

if not os.path.exists(checkpoint_dir):
    raise FileNotFoundError(f"Checkpoint not found: {checkpoint_dir}")

print("Loading checkpoint from: %s" % checkpoint_dir)
param_dict = ms.load_checkpoint(checkpoint_dir)
param_not_load, _ = ms.load_param_into_net(matformer, param_dict)
if param_not_load:
    print("Some parameters were not loaded: %s" % param_not_load)

matformer.set_train(False)  # Switch to inference mode


In [ ]:
def forward_eval(data_x, data_edge_attr, data_edge_index, data_batch, data_label,
                 node_mask, edge_mask, node_num, batch_size):
    pred = matformer(data_x, data_edge_attr, data_edge_index, data_batch,
                     node_mask, edge_mask, node_num)
    mseloss = loss_func_mse(pred, data_label, num=batch_size)
    maeloss = loss_func_mae(pred, data_label, num=batch_size)
    return mseloss, maeloss, pred

eval_step = forward_eval

In [ ]:
# Inference
# Initialize DataLoader
BATCH_SIZE_MAX = config['train']['batch_size']
eval_loader = DataLoader(
    BATCH_SIZE_MAX,
    edge_index_val,
    node_attr=x_val,
    edge_attr=edge_attr_val,
    label=label_val,
    dynamic_batch_size=False,
    shuffle_dataset=False
)

# Store all predictions and labels (for analysis/plotting)
all_preds = []
all_labels = []

eval_mseloss_record = LossRecord()
eval_maeloss_record = LossRecord()

print("Starting inference...")

for batch in eval_loader:
    (node_attr_step, edge_attr_step, label_step, edge_index_step,
     node_batch_step, node_mask_step, edge_mask_step, node_num_step, batch_size_step) = batch

    mseloss_step, maeloss_step, pred_step = eval_step(
        node_attr_step, edge_attr_step, edge_index_step, node_batch_step,
        label_step, node_mask_step, edge_mask_step, node_num_step, batch_size_step
    )

    # Accumulate losses
    eval_mseloss_record.update(mseloss_step)
    eval_maeloss_record.update(maeloss_step)

    # Save predictions and ground truth (convert to numpy for analysis)
    all_preds.append(pred_step.asnumpy())
    all_labels.append(label_step.asnumpy())

    print("Batch MSE: %.6f, MAE: %.6f" % (mseloss_step.asnumpy(), maeloss_step.asnumpy()))

# Concatenate results
all_preds = np.concatenate(all_preds, axis=0)
all_labels = np.concatenate(all_labels, axis=0)

print("Inference completed.")
print("Average MSE: %.6f" % eval_mseloss_record.avg)
print("Average MAE: %.6f" % eval_maeloss_record.avg)


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 6))
plt.scatter(all_labels, all_preds, alpha=0.6)
plt.plot([all_labels.min(), all_labels.max()], [all_labels.min(), all_labels.max()], 'r--')
plt.xlabel('True Value')
plt.ylabel('Predicted Value')
plt.title('Prediction vs True (Validation Set)')
plt.grid(True)
plt.show()

### Visualizing Crystal Graph Structures

In [ ]:
import json
import random

from ase import Atoms
from ase.visualize.plot import plot_atoms

# Parameters
json_path = "jdft_3d-12-12-2022.json"  # Path to dataset

n = 6  # Number of samples to visualize
seed = 42
figsize_each = (4, 4)  # Each subplot size in inches

# Load dataset
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)
total = len(data)
print(f"Dataset loaded with {total} records")

# Randomly select n samples
random.seed(seed)
indices = random.sample(range(total), k=min(n, total))

# Function to build ASE Atoms object from 'atoms' block
def build_ase_atoms_from_atoms_block(atoms_block):
    lattice = np.array(atoms_block["lattice_mat"], dtype=float)
    elements = atoms_block["elements"]
    coords = np.array(atoms_block["coords"], dtype=float)
    cartesian_flag = bool(atoms_block.get("cartesian", True))

    if not cartesian_flag:
        coords = np.dot(coords, lattice)

    ase_atoms = Atoms(symbols=elements, positions=coords, cell=lattice, pbc=True)
    return ase_atoms

# Plot grid
cols = 3
rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * figsize_each[0], rows * figsize_each[1]))
axes = axes.flatten()

for ax_idx, idx in enumerate(indices):
    sample = data[idx]
    jid = sample.get("jid", f"sample_{idx}")
    formula = sample.get("formula", "")
    atoms_block = sample["atoms"]

    try:
        ase_atoms = build_ase_atoms_from_atoms_block(atoms_block)
    except Exception as e:
        print(f"⚠️ Error parsing sample {jid}, skipped: {e}")
        axes[ax_idx].axis("off")
        continue

    plot_atoms(ase_atoms, ax=axes[ax_idx], radii=0.3, rotation="30x,30y,0z")
    axes[ax_idx].set_title(f"{jid}\n{formula}", fontsize=8)
    axes[ax_idx].set_xticks([])
    axes[ax_idx].set_yticks([])

# Turn off remaining empty subplots
for j in range(len(indices), rows * cols):
    axes[j].axis("off")

plt.tight_layout()
plt.show()
